# Uncertainty Engine SDK example workflows - create VariationalGP

This notebook goes through how you would set up a workflow to train and save a VariationalGP model using the `TrainModel` and `Save` node to save this as a Workflow JSON.

In [74]:
from uncertainty_engine import Client
from pprint import pprint

client = Client(
    env="dev"
)
client.authenticate()

In [75]:
my_projects=client.projects.list_projects()


In [76]:
PROJECT_ID = "68dbf8e5c8ed92f3ad5f0a97"

The `ModelConfig` node accepts several optional parameters that control model training. However, as none of the inputs are required we can just use the default input parameters for now. This will configure our model as a `SingleTaskGP` which is ideal for our regression task. A `SingleTaskGP` (Single Task Gaussian Process) models the relationship between a single input and output variable while providing uncertainty estimates alongside predictions.

### Constructing a workflow

First, import and initialise the `Graph` class.


In [77]:
from uncertainty_engine.graph import Graph


# Create a new graph
train_variational_gp_graph = Graph()

In [78]:
from uncertainty_engine.nodes.base import Node

# Define the model config node
model_config = Node(
    node_name="ModelConfig",
    label="Model Config",
)

# Add node to the graph and connect it to the train node
train_variational_gp_graph.add_node(model_config)

# Add a handle to the the config output
output_config = model_config.make_handle("config")

In [ ]:
# Create a LoadDataset node for the x training data
x = Node(
  node_name="LoadDataset",
  label="LoadDatasetX", 
  project_id = PROJECT_ID, 
  file_id = "", 
)

y = Node(
  node_name="LoadDataset",
  label="LoadDatasetY",
  project_id = PROJECT_ID, 
  file_id = "", 
)

train_variational_gp_graph.add_node(x)
train_variational_gp_graph.add_node(y)

In [80]:
# Create handles for the loaded dataset files
x_handle = x.make_handle("file")
y_handle = y.make_handle("file")

# Create the TrainModel node
train_model = Node(
    node_name="TrainModel",
    label="Train Model",
    config=output_config,
    inputs=x_handle.model_dump(),
    outputs=y_handle.model_dump(),
)

# Add the node to the graph
train_variational_gp_graph.add_node(train_model)

In [81]:
output_model = train_model.make_handle("model")

In [ ]:
save_node = Node(
    node_name="Save",
    label="Save Output",
    file=output_model, 
    project_id=PROJECT_ID
)

# Add the save node to your graph
train_variational_gp_graph.add_node(save_node)

Create the executable workflow by wrapping our graph in the `Workflow` node and defining the `requested_output` as the output handle of the `Download` node.

In [85]:
from uncertainty_engine.nodes.workflow import Workflow

pprint(train_variational_gp_graph.nodes) 
# Wrap the graph in a workflow node

train_variational_gp_workflow = Workflow(
    graph=train_variational_gp_graph.nodes,
    inputs=train_variational_gp_graph.external_input,
    external_input_id=train_variational_gp_graph.external_input_id,
    requested_output={}
)


{'nodes': {'LoadDatasetX': {'inputs': {'file_id': {'node_handle': 'LoadDatasetX_file_id',
                                                   'node_name': '_'},
                                       'project_id': {'node_handle': 'LoadDatasetX_project_id',
                                                      'node_name': '_'}},
                            'type': 'LoadDataset'},
           'LoadDatasetY': {'inputs': {'file_id': {'node_handle': 'LoadDatasetY_file_id',
                                                   'node_name': '_'},
                                       'project_id': {'node_handle': 'LoadDatasetY_project_id',
                                                      'node_name': '_'}},
                            'type': 'LoadDataset'},
           'Model Config': {'inputs': {}, 'type': 'ModelConfig'},
           'Save Output': {'inputs': {'file': {'node_handle': 'model',
                                               'node_name': 'Train Model'},
                       

In [89]:
basic_workflow_id = client.workflows.save(
    project_id=PROJECT_ID,
    workflow=train_variational_gp_workflow,
    workflow_name="VariationalGPWorkflow8",
)

Exception: Error creating workflow record: API Error: Bad Request
Details: Duplicate workflow name: A WorkflowRecord with name VariationalGPWorkflow8 already exists in the project.

In [87]:
import json

with open("train_variational_gp_workflow.json", "w") as f:
    json.dump(train_variational_gp_workflow.__dict__, f, indent=2)